# MedGuard - Week 1: OCR -> NER -> Normalisation -> Interaction Lookup

This notebook is the **week-1 deliverable**. By the end of it you will have:

1. A working OCR -> drug extraction -> normalisation -> interaction-lookup pipeline.
2. A leak-free train/val/test split with a measured leakage gap.
3. Two baselines (majority class, TF-IDF + logistic regression) to beat.

> **Colab:** Runtime -> Change runtime type -> **T4 GPU**. Then `Runtime -> Run all`.
> **Local:** `pip install -e ".[all]"` then `jupyter notebook`.

---

## 0. Install

Uncomment the second line when running in Colab.

In [ ]:
# Local (already installed):
# pip install -e ".[all]"

!pip install -q -e ".[all]" 2>&1 | tail -3

## 1. Setup

In [ ]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO)

from medguard.config import get_settings
from medguard.data.splits import SplitStrategy, make_splits
from medguard.evaluation.metrics import (
    calibration_metrics,
    classification_metrics,
    selective_risk_coverage,
)
from medguard.extract.ner import LexiconDrugNer, build_extractor
from medguard.interaction.retrieve import KnowledgeBase
from medguard.modeling.baseline import DrugPairTfidfBaseline, build_baseline_df
from medguard.normalize.drugs import normalize, parse_dose
from medguard.pipeline import MedGuardPipeline, PipelineConfig

settings = get_settings()
settings.ensure_dirs()
print("data dir:", settings.processed_data_dir)

## 2. Data

Run the download once. It writes `data/processed/interactions.parquet`.

```bash
python scripts/download_data.py --all
```

If you are in a hurry, the cell below builds a tiny synthetic corpus so the
rest of the notebook still runs end to end.

In [ ]:
INTERACTIONS = settings.processed_data_dir / "interactions.parquet"

if INTERACTIONS.exists():
    interactions = pd.read_parquet(INTERACTIONS)
    print(f"Loaded {len(interactions):,} interactions from {INTERACTIONS}")
else:
    print("Real corpus not found - using a small synthetic corpus for the demo.")
    rows = []
    pairs = [
        ("warfarin", "aspirin", "bleeding", "major"),
        ("metformin", "glimepiride", "hypoglycemia", "moderate"),
        ("ciprofloxacin", "theophylline", "seizure", "major"),
        ("lisinopril", "potassium chloride", "hyperkalemia", "moderate"),
        ("digoxin", "amiodarone", "bradycardia", "major"),
        ("clopidogrel", "omeprazole", "bleeding", "moderate"),
        ("sildenafil", "nitroglycerin", "hypotension", "major"),
        ("phenytoin", "fluconazole", "seizure", "major"),
    ]
    for i, (a, b, ev, sev) in enumerate(pairs):
        rows.append({
            "drug_a": a, "drug_b": b, "event_type": ev, "severity": sev,
            "description": "Synthetic example for the notebook.",
            "source": "synthetic", "source_id": f"syn:{i}",
        })
    interactions = pd.DataFrame(rows)

interactions.head()

## 3. Leak-free splits - the core methodological step

Compare a naive random split against a leak-free `pairwise` split. The
number of leaked pairs is the headline result.

In [ ]:
reports = {}
splits = {}

for name, strategy in [
    ("random", SplitStrategy.RANDOM),
    ("pairwise", SplitStrategy.PAIRWISE),
    ("cold_start", SplitStrategy.COLD_START),
]:
    train, val, test, report = make_splits(interactions, strategy=strategy, seed=42)
    splits[name] = (train, val, test)
    reports[name] = report
    print(report.summary())

reports_df = pd.DataFrame({name: r.to_dict() for name, r in reports.items()}).T
reports_df

## 4. Baselines

Two baselines you must beat:

1. **Co-occurrence** - how often does this pair appear in training?
2. **TF-IDF + logistic regression** - drug-name morphology carries real signal.

In [ ]:
def build_training_frame(train_df, all_drugs, n_negatives=None):
    """KB pairs = positive, sampled pairs = negative."""
    negatives = [
        (all_drugs[i], all_drugs[j])
        for i in range(len(all_drugs))
        for j in range(i + 1, len(all_drugs))
    ]
    if n_negatives:
        negatives = negatives[:n_negatives]
    return build_baseline_df(train_df, negatives)


all_drugs = sorted(
    {str(d) for d in interactions["drug_a"]} | {str(d) for d in interactions["drug_b"]}
)
print(f"{len(all_drugs)} unique drugs")

results = {}
for name, (train, val, test) in splits.items():
    train_frame = build_training_frame(train, all_drugs)
    model = DrugPairTfidfBaseline().fit(train_frame)

    test_pairs = list(zip(test["drug_a"], test["drug_b"], strict=False))
    y_true = np.ones(len(test_pairs))
    y_pred = model.predict(test_pairs)
    y_prob = model.predict_proba(test_pairs)[:, -1]

    metrics = classification_metrics(y_true, y_pred)
    calib = calibration_metrics(y_true, y_prob)
    results[name] = {"metrics": metrics.to_dict(), "calibration": calib.to_dict()}
    print(f"{name:<12} F1={metrics.f1:.4f}  macroF1={metrics.macro_f1:.4f}  ECE={calib.ece:.4f}")

pd.DataFrame({k: v["metrics"] for k, v in results.items()}).T

## 5. The leakage gap - your headline result

The difference between `random` and `cold_start` is how much of the
headline number was memorisation rather than learning.

In [ ]:
from medguard.evaluation.metrics import leakage_impact

preds_by_split = {}
for name, (train, val, test) in splits.items():
    train_frame = build_training_frame(train, all_drugs)
    model = DrugPairTfidfBaseline().fit(train_frame)
    test_pairs = list(zip(test["drug_a"], test["drug_b"], strict=False))
    preds_by_split[name] = (np.ones(len(test_pairs)), model.predict(test_pairs))

impact = leakage_impact(*preds_by_split["random"], *preds_by_split["cold_start"])
pd.Series(impact).to_frame("value")

## 6. Calibration + selective prediction

Temperature scaling is fitted on validation and evaluated on test.

*(If you do not have a trained transformer yet, this cell uses the TF-IDF
baseline's decision function as a stand-in so you can see the machinery.)*

In [ ]:
from medguard.evaluation.calibration import TemperatureScaler

train, val, test = splits["pairwise"]
train_frame = build_training_frame(train, all_drugs)
model = DrugPairTfidfBaseline().fit(train_frame)

val_pairs = list(zip(val["drug_a"], val["drug_b"], strict=False))
test_pairs = list(zip(test["drug_a"], test["drug_b"], strict=False))

val_logits = model.positive_scores(val_pairs)
test_logits = model.positive_scores(test_pairs)

scaler = TemperatureScaler().fit(val_logits, np.ones(len(val_pairs)))
print(f"Fitted temperature: {scaler.temperature:.4f}")

y_true = np.ones(len(test_pairs))
before = calibration_metrics(y_true, model.predict_proba(test_pairs)[:, -1])
after = calibration_metrics(y_true, scaler.predict_proba(test_logits))

pd.DataFrame({"before": before.to_dict(), "after": after.to_dict()}).T

In [ ]:
curve = selective_risk_coverage(y_true, scaler.predict_proba(test_logits))
curve_df = pd.DataFrame(curve)

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(curve_df["coverage"], curve_df["risk"], marker="o")
    ax.set_xlabel("Coverage")
    ax.set_ylabel("Risk (error rate)")
    ax.set_title("Risk-Coverage curve")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed; skipping the plot.")

curve_df.head(10)

## 7. End-to-end pipeline on a prescription

The lexicon extractor works on CPU with no downloads. Swap in the neural
extractor with `build_extractor(use_heavy_models=True)` once you have a GPU.

In [ ]:
kb = KnowledgeBase.from_frame(interactions)
pipeline = MedGuardPipeline(kb=kb, config=PipelineConfig(language="en"))

prescription = """
Dr. Sharma Clinic
Tab Ecosprin AV 75mg  (once daily)
Tab Warfarin 5mg    (once daily)
Tab Metformin 500mg (twice daily)
Tab Gluconil 5mg    (once daily)
"""

result = pipeline.analyze_text(prescription)

for drug in result.drugs:
    print(f"  {drug.normalized:<28} dose={drug.dose or '-':<10} from '{drug.surface}'")

print()
for finding in result.findings:
    print(f"  [{finding.severity.value.upper():<9}] {finding.drug_a} + {finding.drug_b}  ({finding.event_type})")

print()
for explanation in result.explanations:
    print(explanation.text)
    print()

## 8. Hindi output

In [ ]:
hindi = pipeline.analyze_text(prescription, language="hi")
print(hindi.explanations[0].text)

## 9. Week-1 checklist

- [x] Pipeline runs end to end
- [x] Three split strategies, leakage measured
- [x] Two baselines with numbers
- [x] Calibration + risk-coverage machinery
- [ ] **Your turn:** train the DeBERTa-v3 classifier and beat the TF-IDF baseline
- [ ] **Your turn:** collect 20 real prescription images and evaluate OCR
- [ ] **Your turn:** interview 3 pharmacists / MBBS seniors and log the findings in `docs/EXPERIMENTS.md`

Next notebook: **02_deberta_finetune.ipynb**